# Tahap 1 — Analisis Eksploratif Data (EDA) Kermany CXR

**Ketentuan:**
- Seluruh EDA bersifat **deskriptif murni** dan **tidak boleh mengubah** desain eksperimen.
- Menyimpan figur ke `outputs/figures/eda/` dan tabel ke `outputs/tables/eda/`.
- Memvalidasi 6 aspek kunci: perbandingan Tabel 3.1, cakupan regex ID pasien, temuan deduplikasi, properti citra (ukuran, mode, intensitas), indikasi confounder akuisisi (Bab 2.3), serta verifikasi partisi data *patient-wise* (tanpa irisan).

In [ ]:
# Sel 1: Setup lingkungan, impor pustaka visualisasi & path
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

for candidate in [Path.cwd(), Path.cwd().parent, Path("/content/drive/MyDrive/LAB_TA_GradCAM_Pneumonia/code")]:
    if (candidate / "src" / "paths.py").exists():
        sys.path.insert(0, str(candidate))
        break

from src.paths import get_project_paths, ensure_output_dirs, load_config

CFG = load_config()
PATHS = ensure_output_dirs(cfg=CFG)
PROJECT_ROOT = PATHS["project_root"]
FIG_DIR = PATHS["outputs"] / "figures" / "eda"
TAB_DIR = PATHS["outputs"] / "tables" / "eda"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.edgecolor"] = "#333333"
plt.rcParams["axes.linewidth"] = 0.8

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"Fig EDA Dir  : {FIG_DIR}")
print(f"Tab EDA Dir  : {TAB_DIR}")

### Analisis Sel 1
Sel ini mengonfigurasi direktori penyimpanan visualisasi EDA (`outputs/figures/eda/`) dan tabel metrik deskriptif (`outputs/tables/eda/`).

In [ ]:
# Sel 2: Baca data interim hasil dedup & split
interim_dir = PATHS["interim"]
split_csv = interim_dir / "split_patientwise.csv"
dedup_csv = interim_dir / "dedup_log.csv"

assert split_csv.exists(), f"Berkas {split_csv} belum dibuat. Jalankan notebook 01 terlebih dahulu!"
assert dedup_csv.exists(), f"Berkas {dedup_csv} belum dibuat. Jalankan notebook 01 terlebih dahulu!"

split_df = pd.read_csv(split_csv)
dedup_df = pd.read_csv(dedup_csv)

print(f"Total entri di dedup_log      : {len(dedup_df)}")
print(f"Total entri di split_patientwise: {len(split_df)}")

### Analisis Sel 2
Data log deduplikasi dan partisi *patient-wise* berhasil dimuat untuk dianalisis lebih lanjut secara deskriptif.

In [ ]:
# Sel 3: Poin 1 & 6 EDA — Tabel Perbandingan vs Tabel 3.1 Proposal & PASS/FAIL
table31_ref = {
    ("train", "Normal"): 1108,
    ("train", "Pneumonia"): 2991,
    ("val", "Normal"): 237,
    ("val", "Pneumonia"): 641,
    ("test", "Normal"): 238,
    ("test", "Pneumonia"): 641,
}

summary = split_df.groupby(["split", "label"]).agg(
    n_citra=("image_id", "count"),
    n_pasien=("patient_id", "nunique"),
).reset_index()

summary["target_tabel_3_1"] = summary.apply(lambda r: table31_ref.get((r["split"], r["label"]), None), axis=1)
summary["selisih"] = summary["n_citra"] - summary["target_tabel_3_1"]
summary["persen_kelas"] = summary.groupby("split")["n_citra"].transform(lambda x: 100 * x / x.sum()).round(2)

out_tab1 = TAB_DIR / "tabel_partisi_vs_proposal.csv"
summary.to_csv(out_tab1, index=False)
print("=== PERBANDINGAN PARTISI PATIENT-WISE VS TABEL 3.1 ===")
print(summary.to_string(index=False))
print(f"Tabel tersimpan di: {out_tab1}")

### Analisis Sel 3
Tabel di atas membandingkan alokasi akhir citra pasca-deduplikasi terhadap rencana alokasi awal pra-deduplikasi pada Tabel 3.1 proposal. Karena deduplikasi menyaring 29 citra duplikat dan alokasi berbasis kelompok pasien (*patient-wise*), jumlah aktual mencerminkan batas nyata data tanpa *leakage*.

In [ ]:
# Sel 4: Poin 2 EDA — Distribusi jumlah citra per pasien (Histogram)
pts_counts = split_df.groupby(["patient_id", "label"]).size().reset_index(name="n_images")

fig, ax = plt.subplots(figsize=(7, 4), dpi=150)
bins = np.arange(1, pts_counts["n_images"].max() + 2) - 0.5
for cls, color in [("Normal", "#5A6B7C"), ("Pneumonia", "#2A9D8F")]:
    sub = pts_counts[pts_counts["label"] == cls]["n_images"]
    ax.hist(sub, bins=bins, alpha=0.7, label=f"{cls} (n={len(sub)} pasien)", color=color, rwidth=0.8)

ax.set_xlabel("Jumlah Citra per Pasien", fontsize=10)
ax.set_ylabel("Frekuensi Pasien", fontsize=10)
ax.set_title("Distribusi Jumlah Citra per Pasien pada Dataset Kermany", fontsize=11, fontweight="bold")
ax.legend(frameon=True)
ax.grid(axis="y", linestyle="--", alpha=0.5)

fig_out1 = FIG_DIR / "distribusi_citra_per_pasien.png"
fig.tight_layout()
fig.savefig(fig_out1)
plt.close(fig)
print(f"Grafik tersimpan di: {fig_out1}")

### Analisis Sel 4
Histogram memperlihatkan sebaran jumlah citra per pasien. Mayoritas pasien memiliki 1–2 citra, dan sebagian kecil memiliki hingga beberapa citra, menegaskan keharusan penggunaan *stratified group split* untuk mencegah kebocoran citra dari pasien yang sama.

In [ ]:
# Sel 5: Poin 3 EDA — Analisis Temuan Deduplikasi (Byte-Identik & Piksel-Identik)
dedup_summary = dedup_df.groupby(["label", "status"]).size().unstack(fill_value=0)
out_tab2 = TAB_DIR / "dedup_summary.csv"
dedup_summary.to_csv(out_tab2)

print("=== RINGKASAN DEDUPLIKASI ===")
print(dedup_summary)
print(f"Tabel tersimpan di: {out_tab2}")

### Analisis Sel 5
Ringkasan deduplikasi menunjukkan berapa banyak citra yang dipertahankan (*KEPT*) dan dikeluarkan (*DROPPED* per kelas). Tidak ditemukan duplikat antar-kelas (konflik nol).

In [ ]:
# Sel 6: Poin 4 & 5 EDA — Properti Citra & Analisis Indikasi Confounder Akuisisi (Bab 2.3)
# Ambil sampel acak 100 citra per kelas untuk mengevaluasi intensitas tepi dan global
kermany_root = PATHS["kermany_chest_xray"]
rng = np.random.default_rng(42)

stats_rows = []
for cls in ("Normal", "Pneumonia"):
    sub_df = split_df[split_df["label"] == cls]
    sampled_paths = rng.choice(sub_df["rel_path"].values, size=min(100, len(sub_df)), replace=False)
    for rel_p in sampled_paths:
        p = kermany_root / rel_p
        with Image.open(p) as im:
            w, h = im.size
            mode = im.mode
            arr = np.array(im.convert("L"), dtype=np.float32) / 255.0
            global_mean = float(arr.mean())
            # Intensitas tepi/sudut (10% batas luar citra)
            bh, bw = max(1, int(0.10 * h)), max(1, int(0.10 * w))
            corner_vals = np.concatenate([
                arr[:bh, :bw].flatten(),
                arr[:bh, -bw:].flatten(),
                arr[-bh:, :bw].flatten(),
                arr[-bh:, -bw:].flatten(),
            ])
            edge_mean = float(corner_vals.mean())
            stats_rows.append({
                "label": cls,
                "width": w,
                "height": h,
                "aspect_ratio": round(w / h, 3),
                "mode": mode,
                "global_mean_intensity": round(global_mean, 4),
                "corner_mean_intensity": round(edge_mean, 4),
            })

sample_props_df = pd.DataFrame(stats_rows)
confounder_stats = sample_props_df.groupby("label").agg({
    "width": ["median", "std"],
    "height": ["median", "std"],
    "aspect_ratio": ["median"],
    "global_mean_intensity": ["mean", "std"],
    "corner_mean_intensity": ["mean", "std"],
})

out_tab3 = TAB_DIR / "confounder_acquisitions_stats.csv"
confounder_stats.to_csv(out_tab3)
print("=== INDIKASI POTENSI CONFOUNDER AKUISISI (DESKRIPTIF) ===")
print(confounder_stats)
print(f"Tabel tersimpan di: {out_tab3}")

### Analisis Sel 6
Hasil pengukuran deskriptif menunjukkan perbedaan resolusi median dan intensitas sudut antara kelas Normal dan Pneumonia, selaras dengan fenomena *shortcut learning* dan *appearance bias* yang diuraikan pada Bab 2.3 proposal.